# Pipeline integrado: MIDAGRI + NASA POWER + mapping cultivo-distrito

**Objetivo:** generar `dataset_integrado.csv` y sus insumos intermedios en un solo
flujo reproducible. Consolida los notebooks originales `00_build_mapping_cultivo_distrito`,
`01_midagri_pipeline`, `02_nasa_pipeline` y `03_build_dataset_integrado`
(archivados con el detalle pedagógico celda a celda en `BORRADORES/`).

**Flujo interno:**
```
BDS/<año>/*.xlsx  ──►  [1] MIDAGRI largo      ──►  midagri_largo (en memoria)
                  ──►  [2] Mapping cultivo     ──►  mapping_cultivo_distrito_v2.csv
NASA POWER API    ──►  [3] Descarga clima      ──►  nasa_2020_2025 (en memoria)
                  ──►  [4] Integración JOIN    ──►  dataset_integrado.csv
                                                    dataset_regional.csv
                                                    dataset_por_cultivo.csv
```

**Archivos de salida principales:**
- `OUTPUTS/dataset_integrado.csv` — solo los cultivos más significativos por región: 33 combos (región, cultivo), 28 distritos, ~2 376 filas.
- `OUTPUTS/dataset_regional.csv` — producción agregada por piso ecológico, sin filtro de significancia.
- `BDS/mapping/mapping_cultivo_distrito_v2.csv` — tabla de asignación cultivo → distrito proxy con justificación.

**Incluye el refinamiento SISAGRI** (34 distritos NASA, 38 overrides de mapping) basado en
producción medida 2020-2025 por distrito, no en agronomía general inferida.

## 0. Configuración e importaciones

Detecta `ROOT` del proyecto de forma robusta: si el kernel se lanzó desde
`SCRIPTS/notebooks/`, sube dos niveles; si se lanzó desde `SCRIPTS/`, sube uno.
Esto permite ejecutar el notebook tanto desde VS Code como desde Jupyter Lab sin
cambiar rutas manualmente.

**Constantes clave:**
- `BDS` — carpeta con los Excel de MIDAGRI (`BDS/2020/`, `BDS/2021/`, …).
- `OUTPUTS` — destino de todos los CSV que consumen los notebooks posteriores.
- `ANIOS` — ventana temporal 2020-2025; añadir un año aquí lo propaga a todo el pipeline.
- `REGIONES_OBJETIVO` — las 6 regiones del estudio; cualquier región fuera de esta lista se descarta al cargar MIDAGRI.

**Funciones auxiliares:**
- `normalizar_nombre(s)` — convierte texto a `snake_case` sin tildes ni símbolos. Se aplica a los nombres de columna de cada hoja Excel para homologar variantes tipográficas entre boletines de distintos años.
- `normalizar_region(s)` — versión más simple: solo elimina tildes, conserva mayúsculas y espacios, para poder comparar regiones contra `REGIONES_OBJETIVO`.
- `_extraer_mes_de_filename(filename)` — infiere el mes de un archivo MIDAGRI buscando el nombre del mes en español dentro del nombre del archivo (p. ej. `"Boletin_AGOSTO_2023.xlsx"` → `8`).

In [1]:
from __future__ import annotations

import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent
BDS = ROOT / "BDS"
OUTPUTS = ROOT / "OUTPUTS"
MAPPING_DIR = BDS / "mapping"

ANIOS = [2020, 2021, 2022, 2023, 2024, 2025]
REGIONES_OBJETIVO = ["Piura", "La Libertad", "Ica", "San Martin", "Junin", "Puno"]

MES_NUM = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "SETIEMBRE": 9,
    "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12,
}
NUM_MES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}


def normalizar_nombre(s: str) -> str:
    """ASCII snake_case sin tildes ni simbolos."""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", errors="ignore").decode()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")


def normalizar_region(s: str) -> str:
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", errors="ignore").decode()


def _extraer_mes_de_filename(filename: str) -> int | None:
    nombre = re.sub(r"\s+", " ", Path(filename).stem.upper()).strip()
    return next((num for mes_str, num in MES_NUM.items() if mes_str in nombre), None)


print(f"ROOT: {ROOT}")

ROOT: C:\Users\Joyssie\Documents\dm


## 1. MIDAGRI → `midagri_largo.csv`

Los boletines c-18 de MIDAGRI reportan **producción acumulada año-a-la-fecha (YTD)**,
no producción mensual. Para obtener la producción de cada mes se aplica `diff()`
dentro de cada grupo `(región, año)`: el valor de febrero es YTD_feb − YTD_ene, etc.
Enero es la excepción: el acumulado de enero equivale directamente a la producción de ese mes.

**Casos especiales manejados:**

| Caso | Solución |
|---|---|
| Diciembre-2020 sin boletín propio | Se lee el comparativo incluido en el boletín de diciembre-2021, que publica cifras de diciembre-2020 como año anterior. |
| Mayo-2021 y marzo-2022 sin boletín | Quedan como `NaN` explícito en la grilla completa. |
| Diffs negativos (revisiones retroactivas) | MIDAGRI a veces corrige meses anteriores en boletines posteriores; si la diferencia es negativa no es producción real sino un ajuste contable, por eso se fuerza a `NaN` en lugar de un valor negativo inventado. |

**Política de NaN:** nunca se imputan ceros. Un `NaN` en `produccion_ton` significa
"dato no disponible en esa región/mes", y los notebooks de análisis lo tratan como tal.

**Salida:** DataFrame largo con columnas `[region, anio, mes_num, mes_nombre, cultivo, produccion_mensual]`,
una fila por cada combinación posible de la grilla completa (6 regiones × 6 años × 12 meses × N cultivos).

In [2]:
MAPEO_CANONICO = {
    "arveja_seca": "arveja_grano_seco",
    "haba_seca": "haba_grano_seco",
    "espa_rrago": "esparrago",
    "alca_chofa": "alcachofa",
    "zana_horia": "zanahoria",
    "pallar_gr_seco": "pallar_grano_seco",
    "arveja_gr_seco": "arveja_grano_seco",
    "haba_gr_seco": "haba_grano_seco",
    "maiz_a_duro": "maiz_amarillo_duro",
    "acei_tuna": "aceituna",
}


def _descubrir_archivos_midagri(ruta_base: Path) -> dict[tuple[int, int], Path]:
    """Indexa BDS/<anio>/*.xlsx por (anio, mes), inferido del nombre del archivo."""
    archivos: dict[tuple[int, int], Path] = {}
    for anio in ANIOS:
        carpeta = ruta_base / str(anio)
        if not carpeta.exists():
            continue
        for archivo in carpeta.glob("*.xlsx"):
            mes = _extraer_mes_de_filename(archivo.name)
            if mes is not None:
                archivos[(anio, mes)] = archivo
    return archivos


def _cargar_archivo_midagri(ruta: Path, anio_archivo: int, mes_archivo: int, hoja: str = "c-18") -> pd.DataFrame:
    """Carga un boletin c-18: detecta el header, elimina columnas espejo y
    totales nacionales, normaliza nombres de cultivo y filtra region/anio."""
    df_tmp = pd.read_excel(ruta, sheet_name=hoja, header=None)
    fila_header = next(
        (i for i, row in df_tmp.iterrows()
         if row.astype(str).str.contains("Anos|A\u00f1os", case=False, na=False, regex=True).any()),
        None,
    )
    if fila_header is None:
        raise ValueError(f"Header 'Anos' no encontrado en {ruta}")

    df = pd.read_excel(ruta, sheet_name=hoja, skiprows=fila_header).dropna(how="all").reset_index(drop=True)
    df["Regi\u00f3n"] = df["Regi\u00f3n"].ffill()
    df = df[df["Regi\u00f3n"] != "Total Nacional"]
    df["A\u00f1os"] = df["A\u00f1os"].astype(str).str.extract(r"(\d{4})")[0].astype(float)
    df = df.dropna(subset=["A\u00f1os"])
    df["A\u00f1os"] = df["A\u00f1os"].astype(int)

    cols_espejo = [c for c in df.columns if re.match(r"(Regi\u00f3n|A\u00f1os)\.\d+", str(c))]
    df = df.drop(columns=cols_espejo)

    rename_map = {}
    for col in df.columns:
        if col in ("Regi\u00f3n", "A\u00f1os"):
            rename_map[col] = "region" if col == "Regi\u00f3n" else "anio"
        else:
            norm = normalizar_nombre(col)
            rename_map[col] = MAPEO_CANONICO.get(norm, norm)
    df = df.rename(columns=rename_map)

    if df.columns.duplicated().any():
        df = df.groupby(level=0, axis=1).apply(lambda x: x.bfill(axis=1).iloc[:, 0])

    df["region"] = df["region"].apply(normalizar_region)
    df = df[df["region"].isin(REGIONES_OBJETIVO)]
    df = df[df["anio"] == anio_archivo].copy()
    df["mes_num"] = mes_archivo

    cols_cultivo = [c for c in df.columns if c not in ("region", "anio", "mes_num")]
    for c in cols_cultivo:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.reset_index(drop=True)


def build_midagri_largo() -> pd.DataFrame:
    archivos = _descubrir_archivos_midagri(BDS)
    dfs = [_cargar_archivo_midagri(ruta, anio, mes) for (anio, mes), ruta in sorted(archivos.items())]
    df_acumulado = pd.concat(dfs, ignore_index=True)

    if (2021, 12) in archivos:
        df_dic20 = _cargar_archivo_midagri(archivos[(2021, 12)], anio_archivo=2020, mes_archivo=12)
        mask = (df_acumulado["anio"] == 2020) & (df_acumulado["mes_num"] == 12)
        df_acumulado = pd.concat([df_acumulado[~mask], df_dic20], ignore_index=True)

    grid = pd.MultiIndex.from_product(
        [REGIONES_OBJETIVO, ANIOS, range(1, 13)], names=["region", "anio", "mes_num"]
    ).to_frame(index=False)
    df_acumulado = grid.merge(df_acumulado, on=["region", "anio", "mes_num"], how="left")
    df_acumulado = df_acumulado.sort_values(["region", "anio", "mes_num"]).reset_index(drop=True)

    cols_cultivo = [c for c in df_acumulado.columns if c not in ("region", "anio", "mes_num")]
    df_mensual = df_acumulado.copy()
    for col in cols_cultivo:
        df_mensual[col] = df_acumulado.groupby(["region", "anio"])[col].diff()
        es_enero = df_acumulado["mes_num"] == 1
        df_mensual.loc[es_enero, col] = df_acumulado.loc[es_enero, col]
        df_mensual.loc[df_mensual[col] < 0, col] = np.nan

    df_largo = df_mensual.melt(
        id_vars=["region", "anio", "mes_num"], value_vars=cols_cultivo,
        var_name="cultivo", value_name="produccion_mensual",
    )
    df_largo["mes_nombre"] = df_largo["mes_num"].map(NUM_MES)
    df_largo = df_largo[["region", "anio", "mes_num", "mes_nombre", "cultivo", "produccion_mensual"]]
    df_largo = df_largo.sort_values(["region", "anio", "mes_num", "cultivo"]).reset_index(drop=True)
    return df_largo


df_midagri = build_midagri_largo()
print(f"midagri_largo: {len(df_midagri):,} filas | {df_midagri['cultivo'].nunique()} cultivos | "
      f"{df_midagri['produccion_mensual'].isna().sum()} NaN explicitos")
df_midagri.head()

C:\Users\Joyssie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\Joyssie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\Joyssie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\Joyssie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will

midagri_largo: 23,328 filas | 54 cultivos | 1553 NaN explicitos


,region,anio,mes_num,mes_nombre,cultivo,produccion_mensual
0,Ica,2020,1,Enero,aceituna,0.00
1,Ica,2020,1,Enero,aji,635.00
2,Ica,2020,1,Enero,ajo,12.00
3,Ica,2020,1,Enero,alcachofa,0.00
4,Ica,2020,1,Enero,alfalfa,10684.89


## 2. Mapping cultivo → distrito climático

NASA POWER entrega clima puntual: necesita coordenadas de un lugar concreto.
Como MIDAGRI reporta a nivel regional (no distrital), este paso asigna a cada
par `(región, cultivo)` un **distrito proxy** cuyo clima represente al entorno
productivo real de ese cultivo en esa región.

**Jerarquía de asignación (se aplica en orden, la primera que aplica gana):**

1. **Exclusión explícita** (`EXCLUIR_EXPLICITO`) — combos con volumen marginal o sin
   aptitud agroclimática en esa región se descartan completamente (p. ej. quinua en Ica).
2. **Override regional** (`OVERRIDES`) — decisión manual basada en agronomía o en
   producción medida SISAGRI. Incluye una justificación textual por cada caso.
3. **Taxonomía por piso ecológico** (`CULTIVO_APTITUD`) — si no hay override,
   se recorre la lista de pisos compatibles del cultivo y se asigna el primer
   piso disponible en la región, con confianza "alta" si es el piso prioritario
   o "media" si es un piso alternativo.

**Refinamiento SISAGRI (2024):** para los 33 combos más significativos se verificó el
distrito de mayor producción medida 2020-2025 usando datos SISAGRI (sistema
de información agraria distrital). 26 de los 33 combos tienen un override SISAGRI
con confianza "alta". Los 7 restantes se mantienen por taxonomía porque SISAGRI
no registra el cultivo a nivel distrital con volumen suficiente.

**Salida:** `BDS/mapping/mapping_cultivo_distrito_v2.csv` con 210 combos,
columnas `[región, cultivo, piso_asignado, distrito, produccion_total_ton, nivel_confianza, regla_asignacion, justificacion]`.

In [3]:
DISTRITOS = [
    {"region": "Ica", "distrito": "Chincha Alta", "piso": "costa"},
    {"region": "La Libertad", "distrito": "Viru", "piso": "costa"},
    {"region": "La Libertad", "distrito": "Huamachuco", "piso": "sierra"},
    {"region": "La Libertad", "distrito": "Cascas", "piso": "yunga"},
    {"region": "Piura", "distrito": "Tambogrande", "piso": "bosque_seco"},
    {"region": "Piura", "distrito": "Sullana", "piso": "valle_chira"},
    {"region": "Piura", "distrito": "Canchaque", "piso": "sierra"},
    {"region": "San Martin", "distrito": "Moyobamba", "piso": "selva_alto_mayo"},
    {"region": "San Martin", "distrito": "Tocache", "piso": "selva_huallaga"},
    {"region": "Junin", "distrito": "Perene", "piso": "selva_alta"},
    {"region": "Junin", "distrito": "Rio Tambo", "piso": "selva_baja"},
    {"region": "Junin", "distrito": "El Tambo", "piso": "sierra"},
    {"region": "Puno", "distrito": "Ilave", "piso": "altiplano_lacustre"},
    {"region": "Puno", "distrito": "Ayaviri", "piso": "puna_alta"},
]

CULTIVO_APTITUD: dict[str, dict] = {
    "esparrago": {"pisos": ["costa"], "categoria": "agroexportacion_costa"},
    "uva": {"pisos": ["yunga", "costa"], "categoria": "frutales"},
    "palta": {"pisos": ["costa", "yunga", "selva_alta", "selva_alto_mayo", "selva_huallaga", "valle_chira"], "categoria": "frutales"},
    "mandarina": {"pisos": ["costa", "yunga", "selva_alta", "selva_alto_mayo", "bosque_seco", "valle_chira", "altiplano_lacustre"], "categoria": "citricos"},
    "naranja": {"pisos": ["costa", "yunga", "selva_alta", "selva_alto_mayo", "selva_huallaga", "bosque_seco", "valle_chira", "sierra", "altiplano_lacustre"], "categoria": "citricos"},
    "limon_sutil": {"pisos": ["costa", "bosque_seco", "yunga", "selva_alta", "selva_alto_mayo"], "categoria": "citricos"},
    "tangelo": {"pisos": ["costa", "selva_alta", "selva_alto_mayo"], "categoria": "citricos"},
    "pimiento": {"pisos": ["costa", "valle_chira", "bosque_seco"], "categoria": "hortalizas_exportacion"},
    "piquillo": {"pisos": ["costa", "valle_chira"], "categoria": "hortalizas_exportacion"},
    "paprika": {"pisos": ["costa", "valle_chira"], "categoria": "hortalizas_exportacion"},
    "tomate": {"pisos": ["costa", "yunga", "sierra", "valle_chira", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "hortalizas"},
    "alcachofa": {"pisos": ["costa", "sierra", "altiplano_lacustre"], "categoria": "hortalizas"},
    "cebolla": {"pisos": ["costa", "sierra", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "aji": {"pisos": ["costa", "valle_chira", "yunga", "selva_alta", "selva_alto_mayo", "sierra"], "categoria": "hortalizas"},
    "ajo": {"pisos": ["costa", "sierra", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "zanahoria": {"pisos": ["costa", "sierra", "yunga", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "zapallo": {"pisos": ["costa", "sierra", "yunga", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "arandano": {"pisos": ["costa", "sierra"], "categoria": "frutales_bosque"},
    "mango": {"pisos": ["costa", "bosque_seco", "valle_chira", "selva_alta", "selva_alto_mayo", "sierra"], "categoria": "frutales_tropicales"},
    "papaya": {"pisos": ["costa", "bosque_seco", "valle_chira", "selva_alta", "selva_alto_mayo", "sierra", "altiplano_lacustre"], "categoria": "frutales_tropicales"},
    "platano": {"pisos": ["valle_chira", "bosque_seco", "yunga", "selva_alta", "selva_baja", "selva_huallaga", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "frutales_tropicales"},
    "camote": {"pisos": ["costa", "yunga", "sierra", "valle_chira", "selva_alta", "selva_baja", "selva_huallaga", "altiplano_lacustre"], "categoria": "raices"},
    "yuca": {"pisos": ["costa", "valle_chira", "selva_alta", "selva_baja", "selva_huallaga", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "raices"},
    "maiz_amarillo_duro": {"pisos": ["costa", "valle_chira", "yunga", "selva_alta", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "granos"},
    "maiz_chala": {"pisos": ["costa", "valle_chira", "sierra", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "granos"},
    "maiz_choclo": {"pisos": ["costa", "sierra", "valle_chira", "yunga", "altiplano_lacustre"], "categoria": "granos"},
    "maiz_amilaceo": {"pisos": ["sierra", "yunga", "altiplano_lacustre", "valle_chira"], "categoria": "granos_andinos"},
    "arroz_cascara": {"pisos": ["costa", "valle_chira", "selva_alta", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "granos"},
    "cana_para_azucar": {"pisos": ["costa", "valle_chira"], "categoria": "industrial"},
    "algodon_rama": {"pisos": ["costa", "valle_chira", "selva_alto_mayo"], "categoria": "industrial"},
    "aceituna": {"pisos": ["costa", "valle_chira"], "categoria": "frutales"},
    "pallar_grano_seco": {"pisos": ["costa"], "categoria": "leguminosas"},
    "papa": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre", "yunga"], "categoria": "andinos"},
    "quinua": {"pisos": ["puna_alta", "altiplano_lacustre", "sierra"], "categoria": "andinos"},
    "oca": {"pisos": ["puna_alta", "altiplano_lacustre", "sierra"], "categoria": "andinos"},
    "olluco": {"pisos": ["puna_alta", "altiplano_lacustre", "sierra"], "categoria": "andinos"},
    "trigo": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "granos_andinos"},
    "cebada_grano": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "granos_andinos"},
    "cebada_forrajera": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "forrajeros"},
    "avena_forrajera": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "forrajeros"},
    "alfalfa": {"pisos": ["costa", "sierra", "puna_alta", "altiplano_lacustre", "valle_chira", "selva_alto_mayo"], "categoria": "forrajeros"},
    "haba_grano_seco": {"pisos": ["costa", "sierra", "altiplano_lacustre"], "categoria": "leguminosas"},
    "frijol_grano_seco": {"pisos": ["costa", "yunga", "sierra", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "leguminosas"},
    "arveja_grano_seco": {"pisos": ["sierra", "yunga", "altiplano_lacustre", "costa"], "categoria": "leguminosas"},
    "arveja_verde": {"pisos": ["sierra", "yunga", "altiplano_lacustre", "costa"], "categoria": "leguminosas"},
    "manzana": {"pisos": ["yunga", "sierra"], "categoria": "frutales_templados"},
    "melocoton": {"pisos": ["yunga", "sierra"], "categoria": "frutales_templados"},
    "granadilla": {"pisos": ["yunga", "selva_alta", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "frutales"},
    "tuna": {"pisos": ["costa", "sierra", "yunga", "altiplano_lacustre"], "categoria": "frutales"},
    "oregano": {"pisos": ["sierra", "altiplano_lacustre"], "categoria": "aromaticas"},
    "cafe_pergamino": {"pisos": ["selva_alta", "selva_alto_mayo", "sierra", "altiplano_lacustre"], "categoria": "permanentes_selva"},
    "cacao": {"pisos": ["selva_baja", "selva_huallaga", "selva_alta", "selva_alto_mayo"], "categoria": "permanentes_selva"},
    "pina": {"pisos": ["selva_alta", "selva_alto_mayo", "costa", "altiplano_lacustre"], "categoria": "frutales_tropicales"},
    "palma_aceitera": {"pisos": ["selva_baja", "selva_huallaga"], "categoria": "palma"},
}

OVERRIDES: dict[tuple[str, str], tuple[str, str, str, str]] = {
    ("La Libertad", "uva"): ("yunga", "Cascas", "alta", "Uva de mesa en ceja de selva/yunga (Cascas), no en costa viru."),
    ("La Libertad", "cana_para_azucar"): ("costa", "Viru", "alta", "Agroindustria azucarera concentrada en costa norte (Viru). SISAGRI no registra cana de azucar en La Libertad; sin evidencia medida para refinar, se mantiene este proxy."),
    ("La Libertad", "frijol_grano_seco"): ("yunga", "Cascas", "alta", "Frijol en ceja de selva/yunga (Cascas), coherente con v1 revisado."),
    ("La Libertad", "haba_grano_seco"): ("sierra", "Huamachuco", "alta", "Haba en sierra libertense (Huamachuco)."),
    ("La Libertad", "pina"): ("yunga", "Cascas", "media", "Pina en yunga; costa viru menos representativa."),
    ("Junin", "cacao"): ("selva_baja", "Rio Tambo", "alta", "Cacao en selva baja de la region."),
    ("Junin", "yuca"): ("selva_baja", "Rio Tambo", "alta", "Yuca y cacao comparten selva baja amazonica."),
    ("Junin", "tangelo"): ("selva_alta", "Perene", "alta", "Citricos en ceja de selva (Perene)."),
    ("San Martin", "cacao"): ("selva_huallaga", "Tocache", "alta", "Corredor cacao historico en Tocache."),
    ("Piura", "limon_sutil"): ("bosque_seco", "Tambogrande", "alta", "Limon en bosque seco piurano."),
    ("Piura", "mango"): ("bosque_seco", "Tambogrande", "alta", "Mango en bosque seco."),
    ("Puno", "quinua"): ("puna_alta", "Ayaviri", "alta", "Quinua en puna alta (Ayaviri)."),
    # --- Refinamiento SISAGRI 2020-2025 ---
    ("Ica", "alfalfa"): ("costa", "Independencia", "alta", "SISAGRI: Independencia concentra 235,663 t, mayor produccion medida que Chincha Alta."),
    ("Ica", "esparrago"): ("costa", "Santiago", "alta", "SISAGRI: Santiago concentra 328,506 t, mayor produccion medida que Chincha Alta."),
    ("Ica", "maiz_amarillo_duro"): ("costa", "Independencia", "alta", "SISAGRI: Independencia concentra 350,990 t, mayor produccion medida que Chincha Alta."),
    ("Ica", "mandarina"): ("costa", "El Carmen", "alta", "SISAGRI: El Carmen concentra 422,236 t, mayor produccion medida que Chincha Alta."),
    ("Ica", "palta"): ("costa", "El Carmen", "alta", "SISAGRI: El Carmen concentra 125,142 t, mayor produccion medida que Chincha Alta."),
    ("Ica", "papa"): ("costa", "Nazca", "alta", "SISAGRI: Nazca concentra 242,236 t, reemplaza override anterior de confianza baja."),
    ("Ica", "tomate"): ("costa", "Salas", "alta", "SISAGRI: Salas concentra 201,266 t, mayor produccion medida que Chincha Alta."),
    ("Ica", "uva"): ("costa", "Salas", "alta", "SISAGRI: Salas concentra 799,011 t (cultivo 'VID'), mayor produccion medida que Chincha Alta."),
    ("Junin", "alfalfa"): ("sierra", "Matahuasi", "alta", "SISAGRI: Matahuasi concentra 65,140 t, mayor produccion medida que El Tambo."),
    ("Junin", "avena_forrajera"): ("sierra", "San Jose de Quero", "alta", "SISAGRI: San Jose de Quero concentra 49,383 t, mayor produccion medida que El Tambo."),
    ("Junin", "cafe_pergamino"): ("selva_alta", "Pichanaqui", "alta", "SISAGRI: Pichanaqui concentra 89,394 t, mayor produccion medida que Perene."),
    ("Junin", "maiz_choclo"): ("sierra", "San Agustin", "alta", "SISAGRI: San Agustin concentra 40,617 t, mayor produccion medida que El Tambo."),
    ("Junin", "papa"): ("sierra", "Huasahuasi", "alta", "SISAGRI: Huasahuasi concentra 647,138 t, mayor produccion medida que El Tambo."),
    ("Junin", "pina"): ("selva_alta", "Rio Negro", "alta", "SISAGRI: Rio Negro concentra 690,445 t, mayor produccion medida que Perene."),
    ("Junin", "platano"): ("selva_alta", "Pangoa", "alta", "SISAGRI: Pangoa concentra 266,670 t, mayor produccion medida que Perene."),
    ("La Libertad", "arroz_cascara"): ("costa", "Guadalupe", "alta", "SISAGRI: Guadalupe concentra 380,492 t, mayor produccion medida que Viru."),
    ("Piura", "arroz_cascara"): ("valle_chira", "Ignacio Escudero", "alta", "SISAGRI: Ignacio Escudero concentra 505,128 t, mayor produccion medida que Sullana."),
    ("Piura", "cana_para_azucar"): ("sierra", "Jilili", "media", "SISAGRI solo registra la variante 'CANA DE AZUCAR (ALCOHOL)' (104,422 t en Jilili), producto secundario distinto de la industria azucarera de exportacion dominante en Piura."),
    ("Piura", "platano"): ("valle_chira", "Querecotillo", "alta", "SISAGRI: Querecotillo concentra 380,603 t, reemplaza override anterior."),
    ("Piura", "uva"): ("valle_piura", "Castilla", "alta", "SISAGRI: Castilla concentra 556,876 t (cultivo 'VID'), reemplaza override anterior de confianza media."),
    ("Puno", "alfalfa"): ("altiplano_lacustre", "Taraco", "alta", "SISAGRI: Taraco concentra 888,627 t, reemplaza override anterior."),
    ("Puno", "avena_forrajera"): ("puna_alta", "Pucara", "alta", "SISAGRI: Pucara concentra 1,143,212 t, reemplaza override anterior."),
    ("Puno", "papa"): ("altiplano_lacustre", "Ilave", "alta", "SISAGRI: Ilave concentra 306,830 t, reemplaza override anterior."),
    ("San Martin", "arroz_cascara"): ("selva_huallaga", "Bajo Biavo", "alta", "SISAGRI: Bajo Biavo concentra 662,068 t, mayor produccion medida que Moyobamba."),
    ("San Martin", "maiz_amarillo_duro"): ("selva_huallaga", "Bajo Biavo", "alta", "SISAGRI: Bajo Biavo concentra 188,958 t, mayor produccion medida que Moyobamba."),
    ("San Martin", "platano"): ("selva_alto_mayo", "Moyobamba", "alta", "SISAGRI: Moyobamba concentra 178,762 t, reemplaza override anterior."),
}

EXCLUIR_EXPLICITO: dict[tuple[str, str], str] = {
    ("Ica", "quinua"): "Volumen marginal (<200 t); sin piso andino en distritos Ica.",
    ("Ica", "oca"): "Sin aptitud costera; volumen residual.",
    ("Ica", "olluco"): "Sin aptitud costera; volumen residual.",
    ("Puno", "cacao"): "Cacao no es cultivo de altiplano; probable registro residual.",
    ("La Libertad", "cacao"): "Sin volumen agronomico relevante en costa/sierra libertense.",
}


def _construir_inventario_region() -> dict[str, dict[str, str]]:
    inv: dict[str, dict[str, str]] = {}
    for d in DISTRITOS:
        inv.setdefault(d["region"], {})[d["piso"]] = d["distrito"]
    return inv


def _asignar_distrito(region: str, cultivo: str, produccion: float, inventario: dict) -> dict | None:
    """Prioridad: exclusion explicita > override regional > taxonomia por piso."""
    key = (region, cultivo)
    if key in EXCLUIR_EXPLICITO:
        return None
    if key in OVERRIDES:
        piso, distrito, conf, justificacion = OVERRIDES[key]
        return {
            "region": region, "cultivo": cultivo, "piso_asignado": piso, "distrito": distrito,
            "produccion_total_ton": round(produccion, 2), "nivel_confianza": conf,
            "regla_asignacion": "override_regional",
            "categoria_agronomica": CULTIVO_APTITUD.get(cultivo, {}).get("categoria", "sin_clasificar"),
            "justificacion": justificacion,
        }
    apt = CULTIVO_APTITUD.get(cultivo)
    if apt is None:
        return None
    pisos_region = inventario.get(region, {})
    for piso in apt["pisos"]:
        if piso in pisos_region:
            conf = "alta" if piso == apt["pisos"][0] else "media"
            return {
                "region": region, "cultivo": cultivo, "piso_asignado": piso,
                "distrito": pisos_region[piso], "produccion_total_ton": round(produccion, 2),
                "nivel_confianza": conf, "regla_asignacion": "taxonomia_piso_prioridad",
                "categoria_agronomica": apt["categoria"],
                "justificacion": f"Primer piso compatible en {region}: {piso} "
                                  f"(prioridad {apt['pisos'].index(piso) + 1}/{len(apt['pisos'])}).",
            }
    return None


def build_mapping(df_midagri: pd.DataFrame) -> pd.DataFrame:
    prod = (
        df_midagri.groupby(["region", "cultivo"], as_index=False)["produccion_mensual"]
        .sum(min_count=1)
        .rename(columns={"produccion_mensual": "produccion_total_ton"})
    )
    prod = prod[prod["produccion_total_ton"] > 0]
    inventario = _construir_inventario_region()

    asignados = []
    for _, row in prod.sort_values(["region", "cultivo"]).iterrows():
        res = _asignar_distrito(row["region"], row["cultivo"], row["produccion_total_ton"], inventario)
        if res:
            asignados.append(res)
    return pd.DataFrame(asignados).sort_values(["region", "cultivo"]).reset_index(drop=True)


def exportar_mapping(df_map: pd.DataFrame) -> None:
    MAPPING_DIR.mkdir(parents=True, exist_ok=True)
    df_map.to_csv(MAPPING_DIR / "mapping_cultivo_distrito_v2.csv", index=False, encoding="utf-8-sig")
    df_map[["region", "cultivo", "piso_asignado", "distrito"]].to_csv(
        MAPPING_DIR / "mapping_cultivo_distrito_v2_pipeline.csv", index=False, encoding="utf-8-sig"
    )


df_mapping = build_mapping(df_midagri)
exportar_mapping(df_mapping)
print(f"mapping: {len(df_mapping)} combos | {(df_mapping['nivel_confianza'] == 'alta').sum()} alta confianza | "
      f"{(df_mapping['regla_asignacion'] == 'override_regional').sum()} overrides")
df_mapping.head()

mapping: 210 combos | 114 alta confianza | 38 overrides


,region,cultivo,piso_asignado,distrito,produccion_total_ton,nivel_confianza,regla_asignacion,categoria_agronomica,justificacion
0,Ica,aceituna,costa,Chincha Alta,30373.23,alta,taxonomia_piso_prioridad,frutales,Primer piso compatible en Ica: costa (priorida...
1,Ica,aji,costa,Chincha Alta,46707.77,alta,taxonomia_piso_prioridad,hortalizas,Primer piso compatible en Ica: costa (priorida...
2,Ica,ajo,costa,Chincha Alta,184.83,alta,taxonomia_piso_prioridad,hortalizas,Primer piso compatible en Ica: costa (priorida...
3,Ica,alcachofa,costa,Chincha Alta,210549.20,alta,taxonomia_piso_prioridad,hortalizas,Primer piso compatible en Ica: costa (priorida...
4,Ica,alfalfa,costa,Independencia,757682.18,alta,override_regional,forrajeros,"SISAGRI: Independencia concentra 235,663 t, ma..."


## 3. NASA POWER → `nasa_2020_2025.csv`

[NASA POWER](https://power.larc.nasa.gov/) es una API gratuita que entrega
datos climáticos reanalizados (satélite + modelo atmosférico) en cualquier
coordenada del planeta, con resolución mensual y cobertura desde 1984.

**Parámetros descargados (12 variables):**

| Variable NASA | Nombre final | Unidad |
|---|---|---|
| `T2M` | `temp_promedio` | °C — grados Celsius |
| `T2M_MAX` | `temp_maxima` | °C — grados Celsius |
| `T2M_MIN` | `temp_minima` | °C — grados Celsius |
| `PRECTOTCORR` | `precipitacion` | mm/día — milímetros por día |
| `RH2M` | `humedad_relativa` | % — porcentaje |
| `ALLSKY_SFC_SW_DWN` | `radiacion_solar` | MJ/m²/día — megajoules por metro cuadrado por día |
| `WS2M` | `velocidad_viento` | m/s — metros por segundo |
| `PS` | `presion_atmosferica` | kPa — kilopascales |
| `GWETROOT` | `humedad_suelo` | índice 0–1 adimensional (0 = suelo seco, 1 = suelo saturado) |
| `TS` | `temp_superficie` | °C — grados Celsius |
| `T2MDEW` | `punto_rocio` | °C — grados Celsius |
| `QV2M` | `humedad_especifica` | g/kg — gramos de vapor de agua por kilogramo de aire húmedo |

**Decisiones de limpieza:**
- `MM=13` — NASA incluye una fila de resumen anual con mes=13; se elimina antes de cualquier cálculo.
- Sentinel `-999` — valor centinela de NASA para datos faltantes; se reemplaza por `NaN`.
- **Interpolación lineal por distrito** — si quedan `NaN` tras el reemplazo del centinela, se interpolan dentro de la serie temporal de ese distrito. Se aplica `ffill().bfill()` para cubrir extremos. Al finalizar la descarga debe haber 0 NaN.

**34 distritos:** 14 originales de la versión inicial del mapping + 20 añadidos
con el refinamiento SISAGRI. La descarga es secuencial con reintentos y backoff
exponencial para tolerar timeouts de la API.

In [4]:
URL_NASA = "https://power.larc.nasa.gov/api/temporal/monthly/point"
PARAMETROS_NASA = [
    "T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "ALLSKY_SFC_SW_DWN",
    "WS2M", "PS", "GWETROOT", "TS", "T2MDEW", "QV2M",
]

DISTRITOS_NASA = [
    {"region": "Ica", "distrito": "Chincha Alta", "lat": -13.4099, "lon": -76.1324, "piso": "costa"},
    {"region": "La Libertad", "distrito": "Viru", "lat": -8.4143, "lon": -78.7524, "piso": "costa"},
    {"region": "La Libertad", "distrito": "Huamachuco", "lat": -7.8121, "lon": -78.0487, "piso": "sierra"},
    {"region": "La Libertad", "distrito": "Cascas", "lat": -7.4797, "lon": -78.8178, "piso": "yunga"},
    {"region": "Piura", "distrito": "Tambogrande", "lat": -4.9269, "lon": -80.3447, "piso": "bosque_seco"},
    {"region": "Piura", "distrito": "Sullana", "lat": -4.9039, "lon": -80.6853, "piso": "valle_chira"},
    {"region": "Piura", "distrito": "Canchaque", "lat": -5.3760, "lon": -79.6098, "piso": "sierra"},
    {"region": "San Martin", "distrito": "Moyobamba", "lat": -6.0344, "lon": -76.9742, "piso": "selva_alto_mayo"},
    {"region": "San Martin", "distrito": "Tocache", "lat": -8.1877, "lon": -76.5205, "piso": "selva_huallaga"},
    {"region": "Junin", "distrito": "Perene", "lat": -10.9489, "lon": -75.2264, "piso": "selva_alta"},
    {"region": "Junin", "distrito": "Rio Tambo", "lat": -11.1656, "lon": -74.2353, "piso": "selva_baja"},
    {"region": "Junin", "distrito": "El Tambo", "lat": -12.0313, "lon": -75.2222, "piso": "sierra"},
    {"region": "Puno", "distrito": "Ilave", "lat": -16.0866, "lon": -69.6354, "piso": "altiplano_lacustre"},
    {"region": "Puno", "distrito": "Ayaviri", "lat": -14.8864, "lon": -70.5889, "piso": "puna_alta"},
    {"region": "Ica", "distrito": "Salas", "lat": -13.9214, "lon": -75.7681, "piso": "costa"},
    {"region": "Ica", "distrito": "El Carmen", "lat": -13.4983, "lon": -76.0356, "piso": "costa"},
    {"region": "Ica", "distrito": "Independencia", "lat": -13.6917, "lon": -76.0042, "piso": "costa"},
    {"region": "Ica", "distrito": "Santiago", "lat": -14.1925, "lon": -75.7119, "piso": "costa"},
    {"region": "Ica", "distrito": "Nazca", "lat": -14.8308, "lon": -74.9381, "piso": "costa"},
    {"region": "Piura", "distrito": "Castilla", "lat": -5.1953, "lon": -80.6053, "piso": "valle_piura"},
    {"region": "Piura", "distrito": "Ignacio Escudero", "lat": -4.8647, "lon": -80.8697, "piso": "valle_chira"},
    {"region": "Piura", "distrito": "Querecotillo", "lat": -4.8364, "lon": -80.6456, "piso": "valle_chira"},
    {"region": "Piura", "distrito": "Jilili", "lat": -4.6192, "lon": -79.6108, "piso": "sierra"},
    {"region": "Junin", "distrito": "Rio Negro", "lat": -11.1969, "lon": -74.6542, "piso": "selva_alta"},
    {"region": "Junin", "distrito": "Huasahuasi", "lat": -11.2675, "lon": -75.6469, "piso": "sierra"},
    {"region": "Junin", "distrito": "Pangoa", "lat": -11.4319, "lon": -74.4914, "piso": "selva_alta"},
    {"region": "Junin", "distrito": "Matahuasi", "lat": -11.9161, "lon": -75.3183, "piso": "sierra"},
    {"region": "Junin", "distrito": "San Jose de Quero", "lat": -12.0169, "lon": -75.6264, "piso": "sierra"},
    {"region": "Junin", "distrito": "San Agustin", "lat": -12.0144, "lon": -75.2392, "piso": "sierra"},
    {"region": "Junin", "distrito": "Pichanaqui", "lat": -10.9250, "lon": -74.8719, "piso": "selva_alta"},
    {"region": "Puno", "distrito": "Pucara", "lat": -15.0456, "lon": -70.3683, "piso": "puna_alta"},
    {"region": "Puno", "distrito": "Taraco", "lat": -15.2958, "lon": -69.9839, "piso": "altiplano_lacustre"},
    {"region": "San Martin", "distrito": "Bajo Biavo", "lat": -7.0675, "lon": -76.4367, "piso": "selva_huallaga"},
    {"region": "La Libertad", "distrito": "Guadalupe", "lat": -7.2494, "lon": -79.4756, "piso": "costa"},
]


def _descargar_punto_nasa(distrito_info: dict, max_reintentos: int = 3) -> pd.DataFrame:
    """GET a NASA POWER con reintentos y backoff exponencial."""
    params = {
        "start": "2020", "end": "2025",
        "latitude": distrito_info["lat"], "longitude": distrito_info["lon"],
        "community": "AG", "parameters": ",".join(PARAMETROS_NASA), "format": "JSON",
    }
    for intento in range(1, max_reintentos + 1):
        try:
            r = requests.get(URL_NASA, params=params, timeout=60)
            if r.status_code == 200:
                data = r.json()
                if "properties" not in data or "parameter" not in data["properties"]:
                    raise ValueError(f"Respuesta sin estructura esperada para {distrito_info['distrito']}")
                return pd.DataFrame(data["properties"]["parameter"])
            print(f"  [intento {intento}/{max_reintentos}] HTTP {r.status_code} para {distrito_info['distrito']}")
        except Exception as exc:
            print(f"  [intento {intento}/{max_reintentos}] Error: {exc}")
        if intento < max_reintentos:
            time.sleep(2 ** intento)
    raise RuntimeError(f"Fallo definitivo descargando {distrito_info['distrito']}")


def download_nasa_power() -> pd.DataFrame:
    bloques = []
    for i, dist in enumerate(DISTRITOS_NASA, 1):
        print(f"[nasa] ({i}/{len(DISTRITOS_NASA)}) {dist['region']} - {dist['distrito']}")
        df_raw = _descargar_punto_nasa(dist).reset_index().rename(columns={"index": "fecha_yyyymm"})
        for campo in ("distrito", "region", "piso", "lat", "lon"):
            df_raw[campo] = dist[campo]
        bloques.append(df_raw)

    df_nasa = pd.concat(bloques, ignore_index=True)
    df_nasa = df_nasa[~df_nasa["fecha_yyyymm"].astype(str).str.endswith("13")].copy()
    df_nasa[PARAMETROS_NASA] = df_nasa[PARAMETROS_NASA].replace(-999, np.nan)
    df_nasa["anio"] = df_nasa["fecha_yyyymm"].astype(str).str[:4].astype(int)
    df_nasa["mes_num"] = df_nasa["fecha_yyyymm"].astype(str).str[4:6].astype(int)

    cols_meta = ["distrito", "region", "piso", "lat", "lon", "anio", "mes_num"]
    df_nasa = df_nasa[cols_meta + PARAMETROS_NASA]
    df_nasa = df_nasa.sort_values(["region", "distrito", "anio", "mes_num"]).reset_index(drop=True)
    df_nasa.columns = [c.lower() for c in df_nasa.columns]

    cols_clima = [c.lower() for c in PARAMETROS_NASA]
    for col in cols_clima:
        df_nasa[col] = df_nasa.groupby("distrito")[col].transform(
            lambda x: x.interpolate(method="linear").ffill().bfill()
        )
    return df_nasa


df_nasa = download_nasa_power()
print(f"nasa: {len(df_nasa):,} filas | {df_nasa['distrito'].nunique()} distritos | "
      f"{df_nasa[[c.lower() for c in PARAMETROS_NASA]].isna().sum().sum()} NaN restantes")
df_nasa.head()

[nasa] (1/34) Ica - Chincha Alta
[nasa] (2/34) La Libertad - Viru
[nasa] (3/34) La Libertad - Huamachuco
[nasa] (4/34) La Libertad - Cascas
[nasa] (5/34) Piura - Tambogrande
[nasa] (6/34) Piura - Sullana
[nasa] (7/34) Piura - Canchaque
[nasa] (8/34) San Martin - Moyobamba
[nasa] (9/34) San Martin - Tocache
[nasa] (10/34) Junin - Perene
[nasa] (11/34) Junin - Rio Tambo
[nasa] (12/34) Junin - El Tambo
[nasa] (13/34) Puno - Ilave
[nasa] (14/34) Puno - Ayaviri
[nasa] (15/34) Ica - Salas
[nasa] (16/34) Ica - El Carmen
[nasa] (17/34) Ica - Independencia
[nasa] (18/34) Ica - Santiago
[nasa] (19/34) Ica - Nazca
[nasa] (20/34) Piura - Castilla
[nasa] (21/34) Piura - Ignacio Escudero
[nasa] (22/34) Piura - Querecotillo
[nasa] (23/34) Piura - Jilili
[nasa] (24/34) Junin - Rio Negro
[nasa] (25/34) Junin - Huasahuasi
[nasa] (26/34) Junin - Pangoa
[nasa] (27/34) Junin - Matahuasi
[nasa] (28/34) Junin - San Jose de Quero
[nasa] (29/34) Junin - San Agustin
[nasa] (30/34) Junin - Pichanaqui
[nasa] (31/

,distrito,region,piso,lat,lon,anio,mes_num,t2m,t2m_max,t2m_min,prectotcorr,rh2m,allsky_sfc_sw_dwn,ws2m,ps,gwetroot,ts,t2mdew,qv2m
0,Chincha Alta,Ica,costa,-13.4099,-76.1324,2020,1,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
1,Chincha Alta,Ica,costa,-13.4099,-76.1324,2020,2,22.85,27.98,19.51,0.16,79.22,23.67,2.84,95.77,0.33,24.72,18.86,14.31
2,Chincha Alta,Ica,costa,-13.4099,-76.1324,2020,3,22.62,27.86,19.17,0.20,78.16,22.84,2.70,95.82,0.33,24.34,18.41,13.90
3,Chincha Alta,Ica,costa,-13.4099,-76.1324,2020,4,21.48,27.26,16.69,0.04,73.38,22.14,2.70,95.90,0.33,22.83,16.13,12.10
4,Chincha Alta,Ica,costa,-13.4099,-76.1324,2020,5,20.00,25.60,15.49,0.03,69.33,15.79,2.54,95.90,0.32,20.76,13.80,10.45


## 4. Integración → `dataset_integrado.csv`

Une las tres fuentes por claves compartidas y filtra a los cultivos más significativos por región.

**Lógica de JOIN en dos pasos:**

```
df_midagri  ──[LEFT JOIN por (region, cultivo)]──►  df_mapping
            ──[LEFT JOIN por (distrito, anio, mes_num)]──►  df_nasa
```

El primer join asigna a cada fila de producción su distrito y piso ecológico.
El segundo adjunta las 12 variables climáticas de ese distrito en ese mes concreto.
Dado que cultivos distintos del mismo distrito comparten el mismo clima (vienen
del mismo punto NASA), las variables climáticas se replican por cultivo — esto
es una limitación asumida y documentada en el informe.

**Selección de cultivos significativos (`_filtrar_pareto`):**
Por cada región se ordenan los cultivos de mayor a menor producción acumulada
y se conservan solo los necesarios para alcanzar el 80% del total regional.
Esto reduce ruido de cultivos marginales y es la fuente del número "33 combos"
que aparece en el EDA y el clustering. El umbral 80% es configurable vía
`umbral_pareto`.

**Tres DataFrames de salida:**

| Archivo | Granularidad | Uso |
|---|---|---|
| `dataset_integrado.csv` | (región, cultivo, mes) — solo cultivos significativos | Clustering de perfiles productivos (06b) y EDA por cultivo |
| `dataset_regional.csv` | (región, piso, distrito, mes) — todos | Clustering de zonas agroclimáticas (06a) y EDA regional |
| `dataset_por_cultivo.csv` | (región, cultivo, mes) — todos | Análisis sin filtro cuando se necesita cobertura completa |

In [5]:
RENAME_CLIMA = {
    "t2m": "temp_promedio", "t2m_max": "temp_maxima", "t2m_min": "temp_minima",
    "prectotcorr": "precipitacion", "rh2m": "humedad_relativa",
    "allsky_sfc_sw_dwn": "radiacion_solar", "ws2m": "velocidad_viento",
    "ps": "presion_atmosferica", "gwetroot": "humedad_suelo", "ts": "temp_superficie",
    "t2mdew": "punto_rocio", "qv2m": "humedad_especifica",
}
RENAME_COLUMNAS = {
    **RENAME_CLIMA, "piso_asignado": "piso_ecologico", "piso": "piso_ecologico",
    "mes_num": "numero_mes", "mes_nombre": "mes", "produccion_mensual": "produccion_ton",
    "produccion_total_piso": "produccion_piso_ton", "n_cultivos": "num_cultivos",
}
COLS_META_B = ["region", "piso_asignado", "distrito", "cultivo", "anio", "mes_num", "mes_nombre", "produccion_mensual"]


def _renombrar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={k: v for k, v in RENAME_COLUMNAS.items() if k in df.columns})


def _filtrar_pareto(df_b: pd.DataFrame, umbral: float = 0.80) -> tuple[pd.DataFrame, list[str]]:
    """Conserva, por region, los cultivos necesarios para cubrir >= umbral
    de la produccion regional acumulada (Pareto-80)."""
    combos, reporte = [], []
    for region in sorted(df_b["region"].unique()):
        prod = (
            df_b.loc[df_b["region"] == region].groupby("cultivo")["produccion_mensual"]
            .sum().sort_values(ascending=False)
        )
        prod = prod[prod > 0]
        if prod.empty:
            continue
        acum = prod.cumsum() / prod.sum()
        idx = acum[acum >= umbral].index[0] if (acum >= umbral).any() else acum.index[-1]
        top = acum.loc[:idx].index.tolist()
        combos.extend((region, c) for c in top)
        reporte.append(f"{region:<13}: {len(top)} cultivos ({100 * acum.loc[idx]:.1f}% acumulado)")
    combos_set = set(combos)
    mask = df_b.apply(lambda r: (r["region"], r["cultivo"]) in combos_set, axis=1)
    return df_b.loc[mask].copy(), reporte


def build_dataset_integrado(
    df_midagri: pd.DataFrame, df_nasa: pd.DataFrame, df_mapping: pd.DataFrame, umbral_pareto: float = 0.80
) -> dict[str, pd.DataFrame]:
    mapping = df_mapping[["region", "cultivo", "piso_asignado", "distrito"]].copy()
    df_b = df_midagri.merge(mapping, on=["region", "cultivo"], how="left").dropna(subset=["distrito"]).copy()

    df_nasa_m = df_nasa.rename(columns={"region": "region_nasa"})
    df_b = df_b.merge(df_nasa_m, on=["distrito", "anio", "mes_num"], how="left")
    df_b = df_b.drop(columns=[c for c in ("region_nasa", "lat", "lon", "piso") if c in df_b.columns])
    cols_clima = [c for c in df_b.columns if c not in COLS_META_B]
    df_b = df_b[COLS_META_B + cols_clima]

    df_a_prod = (
        df_b.groupby(["region", "piso_asignado", "distrito", "anio", "mes_num", "mes_nombre"])
        .agg(produccion_total_piso=("produccion_mensual", lambda x: x.sum(min_count=1)),
             n_cultivos=("cultivo", "nunique"))
        .reset_index()
    )
    df_clima = df_b[["distrito", "anio", "mes_num"] + cols_clima].drop_duplicates(["distrito", "anio", "mes_num"])
    df_a = df_a_prod.merge(df_clima, on=["distrito", "anio", "mes_num"], how="left")
    df_a = df_a.rename(columns={"piso_asignado": "piso"})
    cols_meta_a = ["region", "piso", "distrito", "anio", "mes_num", "mes_nombre", "n_cultivos", "produccion_total_piso"]
    df_a = df_a[cols_meta_a + cols_clima]

    df_integrado, reporte_pareto = _filtrar_pareto(df_b, umbral=umbral_pareto)
    print("Pareto-80: " + " | ".join(reporte_pareto))
    return {"dataset_por_cultivo": df_b, "dataset_regional": df_a, "dataset_integrado": df_integrado}


def exportar_datasets(dfs: dict[str, pd.DataFrame]) -> None:
    OUTPUTS.mkdir(parents=True, exist_ok=True)
    for name, df in dfs.items():
        out = _renombrar_columnas(df.copy())
        for col in ("produccion_ton", "produccion_piso_ton"):
            if col in out.columns:
                out[col] = out[col].round(4)
        out.to_csv(OUTPUTS / f"{name}.csv", index=False, encoding="utf-8-sig")

    integrado_path = OUTPUTS / "dataset_integrado.csv"
    if integrado_path.exists():
        (OUTPUTS / "dataset_por_cultivo_filtrado.csv").write_bytes(integrado_path.read_bytes())


dfs = build_dataset_integrado(df_midagri, df_nasa, df_mapping)
exportar_datasets(dfs)

Pareto-80: Ica          : 8 cultivos (80.7% acumulado) | Junin        : 9 cultivos (82.2% acumulado) | La Libertad  : 4 cultivos (81.8% acumulado) | Piura        : 5 cultivos (82.8% acumulado) | Puno         : 3 cultivos (89.9% acumulado) | San Martin   : 4 cultivos (83.8% acumulado)


## 5. Resumen del dataset maestro

Verifica que el CSV final se haya escrito correctamente y muestra las cifras
clave que deben coincidir con las reportadas en el informe:

- **2 376 filas** = 33 combos (región, cultivo) × 12 meses × 6 años (con algunas bajas por NaN de boletines faltantes).
- **20 columnas** = 4 identificadores (región, piso, distrito, cultivo) + 3 temporales (año, mes_num, mes) + 1 producción + 12 variables climáticas.
- **166 NaN en producción** corresponden a los meses sin boletín MIDAGRI (mayo-2021, marzo-2022) y a diciembre-2020 de cultivos sin dato en el comparativo de diciembre-2021.
- **28 distritos únicos** de los 34 descargados de NASA: los 6 restantes quedaron asignados a cultivos fuera del Pareto-80 y no aparecen en el dataset filtrado.

Los notebooks que consumen este archivo son:
- `04_eda.ipynb` — análisis exploratorio
- `06_clustering_final.ipynb` (sección B) — clustering de perfiles productivos
- `06_clustering_final.ipynb` (sección A) — usa `dataset_regional.csv`

In [6]:
df_integrado = pd.read_csv(OUTPUTS / "dataset_integrado.csv")
print(f"dataset_integrado.csv: {len(df_integrado):,} filas | {df_integrado.shape[1]} columnas")
print(f"Combinaciones (region, cultivo): {df_integrado.groupby(['region', 'cultivo']).ngroups}")
print(f"Distritos unicos: {df_integrado['distrito'].nunique()}")
print(f"NaN en produccion_ton: {df_integrado['produccion_ton'].isna().sum()}")
print("\nSiguiente paso: notebooks 04 (EDA) y 06/06a/06b (clustering).")
df_integrado.head()

dataset_integrado.csv: 2,376 filas | 20 columnas
Combinaciones (region, cultivo): 33
Distritos unicos: 28
NaN en produccion_ton: 166

Siguiente paso: notebooks 04 (EDA) y 06/06a/06b (clustering).


,region,piso_ecologico,distrito,cultivo,anio,numero_mes,mes,produccion_ton,temp_promedio,temp_maxima,temp_minima,precipitacion,humedad_relativa,radiacion_solar,velocidad_viento,presion_atmosferica,humedad_suelo,temp_superficie,punto_rocio,humedad_especifica
0,Ica,costa,Independencia,alfalfa,2020,1,Enero,10684.89,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
1,Ica,costa,Santiago,esparrago,2020,1,Enero,14799.60,20.41,27.63,14.85,0.37,69.73,22.02,3.01,87.06,0.39,23.71,14.15,11.72
2,Ica,costa,Independencia,maiz_amarillo_duro,2020,1,Enero,9000.19,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
3,Ica,costa,El Carmen,mandarina,2020,1,Enero,0.00,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
4,Ica,costa,El Carmen,palta,2020,1,Enero,7.10,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
